In [1]:
import pandas as pd
import numpy as np
import itertools
import random

In [2]:
random.seed(42)
np.random.seed(42)

In [3]:
raw_data = pd.read_csv('401k.csv')

In [4]:
# # income, net_tfa, tw quartiles
raw_data['inc_q'], inc_bins = pd.qcut(raw_data['inc'], 4, labels=False, retbins=True)
raw_data['net_tfa_q'], tfa_bins = pd.qcut(raw_data['net_tfa'], 4, labels=False, retbins=True)
raw_data['tw_q'], tw_bins = pd.qcut(raw_data['tw'], 4, labels=False, retbins=True)

# '''
# [ -2652.   19413.   31476.   48583.5 242124. ]
# [-5.023020e+05 -5.000000e+02  1.499000e+03  1.652450e+04  1.536798e+06]
# [-502302.     3291.5   25100.    81487.5 2029910. ]
# '''

In [5]:
# inc_bins = [-3000, 20000, 32000, 50000, 250000]
# tfa_bins = [-60000, 0, 5000, 15000, 200000]
# tw_bins = [-60000, 5000, 25000, 80000, 250000]

In [6]:
inc_means = raw_data.groupby('inc_q')['inc'].mean()
print(inc_means)

inc_q
0    12905.363966
1    25388.084780
2    39277.398305
3    71242.554256
Name: inc, dtype: float64


In [7]:
tfa_means = raw_data.groupby('net_tfa_q')['net_tfa'].mean()
print(tfa_means)

net_tfa_q
0    -7384.978131
1      245.026541
2     6935.821197
3    72532.782574
Name: net_tfa, dtype: float64


In [8]:
tw_means = raw_data.groupby('tw_q')['tw'].mean()
print(tw_means)

tw_q
0     -2758.274708
1     12066.024203
2     48828.306699
3    197125.283985
Name: tw, dtype: float64


In [9]:
def map_quantiles_to_means(data, col, q_col):
    means = data.groupby(q_col)[col].mean()
    return data[q_col].map(means)

# Replace quantile labels with their means
raw_data['inc_q_mean'] = map_quantiles_to_means(raw_data, 'inc', 'inc_q')
raw_data['net_tfa_q_mean'] = map_quantiles_to_means(raw_data, 'net_tfa', 'net_tfa_q')
raw_data['tw_q_mean'] = map_quantiles_to_means(raw_data, 'tw', 'tw_q')

In [10]:
raw_data.head()

,nifa,net_tfa,tw,age,inc,fsize,educ,db,marr,twoearn,e401,p401,pira,hown,inc_q,net_tfa_q,tw_q,inc_q_mean,net_tfa_q_mean,tw_q_mean
0,0.0,0.0,4500.0,47,6765.0,2,8,0,0,0,0,0,0,1,0,1,1,12905.363966,245.026541,12066.024203
1,6215.0,1015.0,22390.0,36,28452.0,1,16,0,0,0,0,0,0,1,1,1,1,25388.084780,245.026541,12066.024203
2,0.0,-2000.0,-2000.0,37,3300.0,6,12,1,0,0,0,0,0,0,0,0,0,12905.363966,-7384.978131,-2758.274708
3,15000.0,15000.0,155000.0,58,52590.0,2,16,0,1,1,0,0,0,1,3,2,3,71242.554256,6935.821197,197125.283985
4,0.0,0.0,58000.0,32,21804.0,1,11,0,0,0,0,0,0,1,1,1,2,25388.084780,245.026541,48828.306699


In [11]:
cols = ['inc_q_mean', 'e401', 'p401', 'net_tfa_q_mean', 'tw_q_mean']
data = raw_data[cols]
data.columns = ['inc_q', 'e401', 'p401', 'net_tfa_q', 'tw_q']

In [12]:
data.head()

,inc_q,e401,p401,net_tfa_q,tw_q
0,12905.363966,0,0,245.026541,12066.024203
1,25388.084780,0,0,245.026541,12066.024203
2,12905.363966,0,0,-7384.978131,-2758.274708
3,71242.554256,0,0,6935.821197,197125.283985
4,25388.084780,0,0,245.026541,48828.306699


In [13]:
def inc_e_prob(data):

    # Get the domain of income and eligibility
    inc_vals = data['inc_q'].unique()
    e_vals = data['e401'].unique()
    ie_vals = list(itertools.product(inc_vals, e_vals))

    # Compute the join probability of income and eligibility
    ie_probs = []
    
    for i, e in ie_vals:
        n = np.sum((data['inc_q'] == i) & (data['e401'] == e)) + 1 # Apply add-one smoothing
        ie_probs.append(n / (data.shape[0] + len(ie_vals)))

    return ie_vals, ie_probs
        
        

In [14]:
def p_fa_prob(data, ie_vals):

    # Unique values of financial assets
    fa_vals = data['net_tfa_q'].unique()

    # U for patiticipation
    pu_vals = ['c', 'n']

    # Probabilities of pu given income and financial assets given income, pu and participation
    p_prob = {}
    fa_probs = {}

    # Unique values of income
    i_vals = data['inc_q'].unique()

    for i in i_vals:

        # Get the data for income i
        di = data[data['inc_q'] == i]
        
        # Compute P(p_0 \mid e_1) and P(p_1 \mid e_1) (e = 0, p = 1) does not exist in the data
        pn = np.sum((di['p401'] == 0) & (di['e401'] == 1)) / np.sum(di['e401'] == 1)
        pc = np.sum((di['p401'] == 1) & (di['e401'] == 1)) / np.sum(di['e401'] == 1)

        assert pn + pc == 1 # Make sure the probabilities sum to 1

        # # Sanity checks
        # p_x0z0 = np.sum((di['p401'] == 0) & (di['e401'] == 0)) / np.sum(di['e401'] == 0)
        # p_x0z1 = np.sum((di['p401'] == 0) & (di['e401'] == 1)) / np.sum(di['e401'] == 1)
        # assert p_x0z0 - p_x0z1 == pc
        
        p_prob[i] = [pc, pn]

        # Compute probability of fa given income, pu and p
        fa_probs[(i, 'c', 0)] = []
        fa_probs[(i, 'c', 1)] = []
        fa_probs[(i, 'n', 0)] = []
        fa_probs[(i, 'n', 1)] = []

        for f in fa_vals:
            pf_x0z1 = np.sum((di['net_tfa_q'] == f) & (di['p401'] == 0) & (di['e401'] == 1)) / np.sum((di['e401'] == 1)) # P(f, p=0 | e=1)
            pf_x1z1 = np.sum((di['net_tfa_q'] == f) & (di['p401'] == 1) & (di['e401'] == 1)) / np.sum((di['e401'] == 1)) # P(f, p=1 | e=1)
            pf_x0z0 = np.sum((di['net_tfa_q'] == f) & (di['p401'] == 0) & (di['e401'] == 0)) / np.sum((di['e401'] == 0)) # P(f, p=0 | e=0)
            pf_x1z0 = np.sum((di['net_tfa_q'] == f) & (di['p401'] == 1) & (di['e401'] == 0)) / np.sum((di['e401'] == 0)) # P(f, p=1 | e=0) = 0

            fa_probs[(i, 'n', 0)].append(pf_x0z1 / p_prob[i][1]) # P(f, p=0 | e=1)/P(p=0 | e=1)
            fa_probs[(i, 'n', 1)].append(1/len(fa_vals)) # Can put any random value here

            fa_probs[(i, 'c', 1)].append(pf_x1z1 / p_prob[i][0]) # P(f, p=1 | e=1)/P(p=1 | e=1)
            fa_probs[(i, 'c', 0)].append((pf_x0z0 - pf_x0z1) / p_prob[i][0]) # (P(f, p=0 | e=0) - P(f, p=0 | e=1))/P(p=1 | e=1)

    return pu_vals, p_prob, fa_vals, fa_probs

In [15]:
def tw_prob(data, ie_vals, pf_vals):

    tw_vals = data['tw_q'].unique()
    tw_probs = {}

    for i, e in ie_vals:
        for p, f in pf_vals:
            
            d = np.sum((data['inc_q'] == i) & (data['e401'] == e) & (data['p401'] == p) & (data['net_tfa_q'] == f)) + len(tw_vals)
            tw_probs[(i, e, p, f)] = []

            for t in tw_vals:
                n = np.sum((data['inc_q'] == i) & (data['e401'] == e) & (data['p401'] == p) & (data['net_tfa_q'] == f) & (data['tw_q'] == t)) + 1

                tw_probs[(i, e, p, f)].append(n / d)

    return tw_vals, tw_probs

In [16]:
def sample(data, n = 10000):

    # Get the domain and distribution of income and eligibility
    ie_vals, ie_probs = inc_e_prob(data)

    # Get the domain and distribution of  pu and financial assets
    pu_vals, p_prob, fa_vals, fa_prob = p_fa_prob(data, ie_vals)
    
    # Domain of participation and financial assets
    pf_vals = list(itertools.product([0, 1], fa_vals))

    # Get the domain and distribution of total wealth
    tw_vals, tw_probs = tw_prob(data, ie_vals, pf_vals)

    samples = []

    for _ in range(n):
        i, e = ie_vals[np.random.choice(len(ie_vals), p = ie_probs)] # Sample income and eligibility

        pu = pu_vals[np.random.choice(len(pu_vals), p = p_prob[i])] # Sample pu
        p = e if pu == 'c' else 0 # Compute participation

        f = fa_vals[np.random.choice(len(fa_vals), p = fa_prob[(i, pu, p)])] # Sample financial assets

        t = np.random.choice(tw_vals, p = tw_probs[(i, e, p, f)]) # Sample total wealth

        samples.append((i, e, p, f, t))

    new_data = pd.DataFrame(samples, columns = ['inc', 'e401', 'p401', 'net_tfa', 'tw'])
    return new_data

In [17]:
new_data = sample(data, 30000)

In [18]:
new_data.shape

(30000, 5)

In [19]:
new_data.head()

,inc,e401,p401,net_tfa,tw
0,25388.084780,0,0,6935.821197,197125.283985
1,12905.363966,0,0,245.026541,197125.283985
2,71242.554256,0,0,245.026541,48828.306699
3,39277.398305,0,0,245.026541,12066.024203
4,25388.084780,0,0,-7384.978131,-2758.274708


In [20]:
new_data.to_csv('401k_sampled.csv', index = False)

In [21]:
# Get samples for the LATE query
def sample_q(data, n = 10000):

    # Get the domain and distribution of income and eligibility
    ie_vals, ie_probs = inc_e_prob(data)

    # Get the domain and distribution of  pu and financial assets
    pu_vals, p_prob, fa_vals, fa_prob = p_fa_prob(data, ie_vals)

    # Domain of participation and financial assets
    pf_vals = list(itertools.product([0, 1], fa_vals))

    # Get the domain and distribution of total wealth
    tw_vals, tw_probs = tw_prob(data, ie_vals, pf_vals)

    # Samples for p = 0
    p0_samples = []
    for _ in range(n):

        i, e = ie_vals[np.random.choice(len(ie_vals), p = ie_probs)] # Sample income and eligibility

        pu = pu_vals[np.random.choice(len(pu_vals), p = p_prob[i])] # Sample pu

        p = 0 # Set participation to 0

        f = fa_vals[np.random.choice(len(fa_vals), p = fa_prob[(i, pu, p)])] # Sample financial assets

        t = np.random.choice(tw_vals, p = tw_probs[(i, e, p, f)]) # Sample total wealth

        p0_samples.append((i, e, pu, f, t))

    p0_data = pd.DataFrame(p0_samples, columns = ['inc', 'e401', 'pu', 'net_tfa', 'tw'])

    p1_samples = []
    for _ in range(n):

        i, e = ie_vals[np.random.choice(len(ie_vals), p = ie_probs)] # Sample income and eligibility

        pu = pu_vals[np.random.choice(len(pu_vals), p = p_prob[i])] # Sample pu
        p = 1 # Set participation to 1

        f = fa_vals[np.random.choice(len(fa_vals), p = fa_prob[(i, pu, p)])] # Sample financial assets

        t = np.random.choice(tw_vals, p = tw_probs[(i, e, p, f)]) # Sample total wealth

        p1_samples.append((i, e, pu, f, t))

    p1_data = pd.DataFrame(p1_samples, columns = ['inc', 'e401', 'pu', 'net_tfa', 'tw'])

    return p0_data, p1_data

In [22]:
p0_data, p1_data = sample_q(data, 500000)

In [23]:
ep0 = p0_data[p0_data['pu'] == 'c'].groupby('inc')['tw'].mean()

In [24]:
ep0

inc
12905.363966     21632.474931
25388.084780     42716.816323
39277.398305     66191.104328
71242.554256    109708.079598
Name: tw, dtype: float64

In [25]:
ep1 = p1_data[p1_data['pu'] == 'c'].groupby('inc')['tw'].mean()

In [26]:
ep1

inc
12905.363966     59726.192009
25388.084780     60186.639494
39277.398305     71086.998391
71242.554256    104111.445791
Name: tw, dtype: float64

In [27]:
ep1 - ep0

inc
12905.363966    38093.717078
25388.084780    17469.823171
39277.398305     4895.894063
71242.554256    -5596.633807
Name: tw, dtype: float64

In [28]:
ed = ep1 - ep0

In [29]:
ed.to_pickle('401k_effect.pkl')